In [1]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("sachinkumar62/phone-price-prediction-dataset")

print("Path to dataset files:", path)

100%|██████████| 12.7M/12.7M [00:00<00:00, 37.0MB/s]

Extracting files...


Path to dataset files: /root/.cache/kagglehub/datasets/sachinkumar62/phone-price-prediction-dataset/versions/2


In [2]:
import os
import pandas as pd
import re
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
import cv2
import requests
import math
from io import BytesIO
import concurrent.futures
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
import keras
from keras import layers

In [3]:
os.listdir(path)

['data']

In [4]:
path_data = os.path.join(path, "data")
path_images = os.path.join(path_data, "downloaded_images")
path_csv = os.path.join(path_data, "dataset.csv")

In [ ]:
dataset = pd.read_csv(path_csv)
dataset.head()

In [ ]:
plt.figure(figsize=(15, 12))
for i in range(9):
    plt.subplot(3, 3, i+1)
    title = dataset["Title"].iloc[i].split("(")[0] + " \nPrice: " + dataset["Price"].iloc[i].replace(",", ".")
    plt.title(title)
    img_url = dataset["Image URL"].iloc[i]
    response = requests.get(img_url)
    img = Image.open(BytesIO(response.content))
    plt.imshow(img)
    plt.axis("off")

In [7]:
dataset[dataset["Rating"] == "No Rating"] = np.nan
dataset.dropna(inplace=True)
dataset["Rating"] = dataset["Rating"].astype(float)

In [ ]:
data = dataset.sort_values(by="Rating", ascending=False)
data = data.reset_index(drop=True).head(10)
plt.figure(figsize=(15, 12))
for i in range(9):
    plt.subplot(3, 3, i+1)
    title = data["Title"].iloc[i].split("(")[0] + " \nRating: " + str(data["Rating"].iloc[i])
    plt.title(title)
    img_url = data["Image URL"].iloc[i]
    response = requests.get(img_url)
    img = Image.open(BytesIO(response.content))
    plt.imshow(img)
    plt.axis("off")

In [ ]:
plt.figure(figsize=(12, 12))
for i in range(9):
    plt.subplot(3, 3, i+1)
    image_files = os.listdir(path_images)
    if len(image_files) > 0:
      image_path = os.path.join(path_images, image_files[i])
      if os.path.exists(image_path):
        img = Image.open(image_path)
        plt.imshow(img)
      else:
        print(f"Image file not found: {image_path}")
    else:
        print("No images found in the directory.")
    plt.axis("off")

In [ ]:
dataset["Label"] = dataset["Title"].apply(lambda x: x.split(" ")[0].lower())
dataset.head()

In [ ]:
le = LabelEncoder()
le.fit(dataset['Label'])
dataset['Label'] = le.transform(dataset['Label'])
dataset.head()

In [12]:
classes = dataset["Label"].unique()
labels = dataset["Label"]

In [13]:
def read_image(image_url):
    response = requests.get(image_url, stream=True)
    img = np.array(bytearray(response.content), dtype=np.uint8)
    img = cv2.imdecode(img, cv2.IMREAD_COLOR)
    return img

def load_images_parallel(urls):
    with concurrent.futures.ThreadPoolExecutor() as executor:
        images = list(executor.map(read_image, urls))
    return [img for img in images if img is not None]

def preprocess_image(image, target_size=(224, 224)):

    image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    image_resized = cv2.resize(image_rgb, target_size)
    image_normalized = image_resized / 255.0
    return np.array(image_normalized, dtype=np.float32)

In [14]:
urls = dataset["Image URL"].values
images = load_images_parallel(urls)
images = [preprocess_image(img) for img in images]
images = np.stack(images, axis=0)

In [15]:
X_train, X_test, y_train, y_test = train_test_split(images, labels,
                                                    test_size=0.2,
                                                    random_state=42)

In [16]:
def make_model(input_shape, num_classes, dropout_rate=0.2):
    inputs = keras.Input(shape=input_shape)

    # Bloco inicial com Conv2D e BatchNorm
    x = layers.Conv2D(32, 3, strides=2, padding="same")(inputs)
    x = layers.BatchNormalization()(x)
    x = layers.Activation("relu")(x)

    x = layers.Conv2D(64, 3, padding="same")(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation("relu")(x)

    previous_block_activation = x  # Salvando ativação anterior para adicionar ao bloco residual

    # Blocos intermediários com SeparableConv2D, BatchNorm e Resíduo
    for size in [128, 256, 512, 728]:
        x = layers.Activation("relu")(x)
        x = layers.SeparableConv2D(size, 3, padding="same")(x)
        x = layers.BatchNormalization()(x)

        x = layers.Activation("relu")(x)
        x = layers.SeparableConv2D(size, 3, padding="same")(x)
        x = layers.BatchNormalization()(x)

        # MaxPooling reduzindo a resolução
        x = layers.MaxPooling2D(3, strides=2, padding="same")(x)

        # Conexão residual
        residual = layers.Conv2D(size, 1, strides=2, padding="same")(previous_block_activation)
        x = layers.add([x, residual])
        previous_block_activation = x  # Atualizando a ativação anterior para o próximo bloco

    # Último bloco convolucional
    x = layers.SeparableConv2D(1024, 3, padding="same")(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation("relu")(x)

    # Pooling global para redução final
    x = layers.GlobalAveragePooling2D()(x)

    if num_classes == 2:
        activation = "sigmoid"
        units = 1
    else:
        activation = "softmax"
        units = num_classes

    x = layers.Dropout(dropout_rate)(x)

    outputs = layers.Dense(units, activation=activation)(x)

    return keras.Model(inputs, outputs)

In [17]:
input_shape = X_train.shape[1:]
num_classes = len(np.unique(classes))

model = make_model(input_shape, num_classes)

In [23]:
model.compile(
    optimizer=keras.optimizers.Adam(1e-3),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)
model.fit(X_train, y_train, batch_size=64, epochs=10,
          validation_data=(X_test, y_test))

Epoch 1/10
20/20 ━━━━━━━━━━━━━━━━━━━━ 34s 919ms/step - accuracy: 0.9667 - loss: 0.1480 - val_accuracy: 0.0422 - val_loss: 4.6428
Epoch 2/10
20/20 ━━━━━━━━━━━━━━━━━━━━ 22s 347ms/step - accuracy: 0.9716 - loss: 0.1320 - val_accuracy: 0.0422 - val_loss: 4.7081
Epoch 3/10
20/20 ━━━━━━━━━━━━━━━━━━━━ 10s 354ms/step - accuracy: 0.9882 - loss: 0.0665 - val_accuracy: 0.0422 - val_loss: 4.7119
Epoch 4/10
20/20 ━━━━━━━━━━━━━━━━━━━━ 10s 355ms/step - accuracy: 0.9907 - loss: 0.0404 - val_accuracy: 0.0422 - val_loss: 4.7166
Epoch 5/10
20/20 ━━━━━━━━━━━━━━━━━━━━ 11s 371ms/step - accuracy: 0.9871 - loss: 0.0566 - val_accuracy: 0.0390 - val_loss: 4.8356
Epoch 6/10
20/20 ━━━━━━━━━━━━━━━━━━━━ 10s 368ms/step - accuracy: 0.9844 - loss: 0.0559 - val_accuracy: 0.0422 - val_loss: 4.8630
Epoch 7/10
20/20 ━━━━━━━━━━━━━━━━━━━━ 10s 368ms/step - accuracy: 0.9897 - loss: 0.0570 - val_accuracy: 0.0422 - val_loss: 5.5435
Epoch 8/10
20/20 ━━━━━━━━━━━━━━━━━━━━ 10s 360ms/step - accuracy: 0.9886 - loss: 0.0431 - val_accu